In [15]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torchmetrics

torch.manual_seed(42)
np.random.seed(42)

data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)

def cyclical(df, col, period):
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

data["hour"] = data["time"].dt.hour
data["month"] = data["time"].dt.month
data["dow"] = data["time"].dt.dayofweek

cyclical(data, "hour", 24)
cyclical(data, "month", 12)
cyclical(data, "dow", 7)
data["wind_dir_sin"] = np.sin(np.deg2rad(data["wind_direction_10m"]))
data["wind_dir_cos"] = np.cos(np.deg2rad(data["wind_direction_10m"]))

feature_cols = [
    "pm2_5", "pm10", "carbon_monoxide", "nitrogen_dioxide", "sulphur_dioxide", "ozone",
    "temperature_2m", "relative_humidity_2m", "wind_speed_10m", "surface_pressure",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "wind_dir_sin", "wind_dir_cos",
]
target_col = "pm2_5"

In [16]:
WINDOW = 12
HORIZON = 1

def create_windows(features, target, window=WINDOW, horizon=HORIZON):
    X, y = [], []
    for i in range(len(features) - window - horizon + 1):
        X.append(features[i:i + window])
        y.append(target[i + window + horizon - 1])
    return np.array(X), np.array(y)

X_raw = data[feature_cols].values
y_raw = data[[target_col]].values
X, y = create_windows(X_raw, y_raw)

# Chronological split: 80% train, 20% test
n = len(X)
train_end = int(0.8 * n)

X_train, X_test = X[:train_end], X[train_end:]
y_train, y_test = y[:train_end], y[train_end:]

feature_scaler = StandardScaler().fit(X_train.reshape(-1, X_train.shape[-1]))
X_train_s = feature_scaler.transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test_s = feature_scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

target_scaler = StandardScaler().fit(y_train)
y_train_s = target_scaler.transform(y_train)
y_test_s = target_scaler.transform(y_test)

def to_dataset(X, y):
    return TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

train_dataset = to_dataset(X_train_s, y_train_s)
test_dataset = to_dataset(X_test_s, y_test_s)

In [17]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

device = "cuda" if torch.cuda.is_available() else "cpu"
n_inputs = train_dataset[0][0].shape[-1]

class LSTMForecaster(nn.Module):
    def __init__(self, n_hidden, n_layers=2, n_inputs=n_inputs, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(n_inputs, n_hidden, n_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_hidden, 1)

    def forward(self, X):
        out, _ = self.lstm(X)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

def evaluate(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred.squeeze(-1), y_batch.squeeze(-1))
    return metric.compute()

In [18]:
import time
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# Hyperparameters
# -----------------------------
N_HIDDEN = 64
DROPOUT = 0.1
LR = 0.001
WEIGHT_DECAY = 1e-4
N_EPOCHS = 20

final_model = LSTMForecaster(
    n_hidden=N_HIDDEN,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
criterion = nn.MSELoss()

# -----------------------------
# Train (fixed epochs, no validation)
# -----------------------------
train_start = time.time()

for epoch in range(N_EPOCHS):
    final_model.train()
    total_loss = 0.

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = final_model(X_batch)
        loss = criterion(y_pred, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    mean_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{N_EPOCHS}, train loss: {mean_loss:.4f}")

training_time = time.time() - train_start

# -----------------------------
# Predict on test set
# -----------------------------
final_model.eval()
all_preds, all_true = [], []

start = time.time()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_pred = final_model(X_batch)
        all_preds.append(y_pred.cpu().numpy())
        all_true.append(y_batch.numpy())
inference_time = (time.time() - start) / len(test_dataset)

preds_scaled = np.concatenate(all_preds)
true_scaled = np.concatenate(all_true)

# -----------------------------
# Inverse transform to original PM2.5 scale
# -----------------------------
preds = target_scaler.inverse_transform(preds_scaled)
true = target_scaler.inverse_transform(true_scaled)

# -----------------------------
# Metrics
# -----------------------------
rmse = root_mean_squared_error(true, preds)
mae = mean_absolute_error(true, preds)
r2 = r2_score(true, preds)

print(f"\nRMSE          : {rmse:.4f}")
print(f"MAE           : {mae:.4f}")
print(f"R²            : {r2:.4f}")
print(f"Training time : {training_time:.2f}s")
print(f"Inference time: {inference_time*1_000_000:.4f}µs")

Epoch 1/20, train loss: 0.1977
Epoch 2/20, train loss: 0.0673
Epoch 3/20, train loss: 0.0567
Epoch 4/20, train loss: 0.0505
Epoch 5/20, train loss: 0.0455
Epoch 6/20, train loss: 0.0435
Epoch 7/20, train loss: 0.0418
Epoch 8/20, train loss: 0.0411
Epoch 9/20, train loss: 0.0395
Epoch 10/20, train loss: 0.0392
Epoch 11/20, train loss: 0.0369
Epoch 12/20, train loss: 0.0375
Epoch 13/20, train loss: 0.0375
Epoch 14/20, train loss: 0.0368
Epoch 15/20, train loss: 0.0357
Epoch 16/20, train loss: 0.0365
Epoch 17/20, train loss: 0.0354
Epoch 18/20, train loss: 0.0346
Epoch 19/20, train loss: 0.0338
Epoch 20/20, train loss: 0.0342

RMSE          : 8.2823
MAE           : 4.5130
R²            : 0.9233
Training time : 136.44s
Inference time: 171.0954µs
